In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd


In [2]:
print("Loading cleaned dataset from local folder.")

df = pd.read_csv("/home/robin/Code_repo/psycholinguistic2125/JADT_rap_fr/data/20260123_filter_verses_lrfaf_corpus.csv")

print(f"Dataset loaded: {len(df)} songs")
print(f"Columns: {df.columns.tolist()}")
print(f"Date range: {df['year'].min()} to {df['year'].max()}")

Loading cleaned dataset from local folder.
Dataset loaded: 115805 songs
Columns: ['Unnamed: 0', 'artist', 'title', 'year', 'lyrics', 'lyrics_cleaned', 'born_in_france', 'url', 'birthdate_artist', 'age_artist', 'IRAMUTEQ_CLASSES']
Date range: 1992.0 to 2023.0


In [3]:
docs = df['lyrics_cleaned'].tolist()

In [5]:
from sentence_transformers import SentenceTransformer
from bunkatopics import Bunka
# Load Projection Model
import umap
import numpy as np

embedding_model = SentenceTransformer("dangvantuan/sentence-camembert-base") #SentenceTransformer('intfloat/multilingual-e5-base')

projection_model = umap.UMAP(
    n_components=2,        # 2D visualization
    n_neighbors=15,        # Balanced local/global
    min_dist=0.1,          # Tight clusters
    metric='cosine',       # For embeddings
    random_state=42)


bunka = Bunka(embedding_model=embedding_model, 
    projection_model=projection_model,
    language="french")  # the language is automatically detected, make sure the embedding model is adapted

# Fit Bunka to your text data
bunka.fit(docs)

bunka.save_bunka("models/bunka_filtered_verses_v1")

2026-01-23 10:52:04 - Bunka - INFO - Processing 31833685 tokens
2026-01-23 10:52:08 - Bunka - INFO - Detected language: French
2026-01-23 10:52:08 - Bunka - INFO - Embedding documents... (can take varying amounts of time depending on their size)


Batches:   0%|          | 0/3619 [00:00<?, ?it/s]

2026-01-23 11:00:03 - Bunka - INFO - Reducing the dimensions of embeddings...
2026-01-23 11:01:51 - Bunka - INFO - Extracting meaningful terms from documents...
2026-01-23 11:01:51 - Bunka - INFO - Sampling 1000 documents for term extraction
100%|██████████| 1000/1000 [00:28<00:00, 34.95it/s]


In [6]:
bunka.get_topics(n_clusters=20,name_length=15)


2026-01-23 11:03:12 - Bunka - INFO - Computing the topics


,topic_id,topic_name,size,percent
0,bt-4,lance | weed | plainte | descend | plans | fus...,7758,6.70
1,bt-12,jaune | pisté | table | clope | boss | fouille...,7453,6.44
2,bt-16,loves | arrête | parlons | soirées | barbe | d...,7428,6.41
3,bt-8,sûrement | boussole | Besoin | piranhas | meur...,7400,6.39
4,bt-10,gérant | mens | lundi | vitre | viens | ciudad...,6921,5.98
5,bt-13,hip | beat | flow | pique | pot | assurance | ...,6502,5.61
6,bt-2,virus | boys | up | rap | mythos | vaccin | pr...,6470,5.59
7,bt-18,plaies | velours | Prie | sentiments | ladies ...,5944,5.13
8,bt-14,sommeil | routine | croix | atmosphère | coté ...,5928,5.12
9,bt-1,froid | shtar | love | foudroie | tournoient |...,5784,4.99


In [ ]:
bunka.visualize_topics(width=800, height=800, colorscale='delta')

2026-01-23 11:03:49 - Bunka - INFO - Creating the Bunka Map


In [ ]:
bunka.df_top_docs_per_topic_#.to_json("models/bunka_rap_v1/most_representativedocs_per_topics.json")

In [19]:
# Option one; # Each element in bunka.docs is a pydantic Document with fields like doc_id, x, y, topic_id, term_id…
docs_with_topics = [doc.model_dump() for doc in bunka.docs]

df_docs_topics = pd.DataFrame(docs_with_topics)

merged = df.merge(df_docs_topics, left_on = 'lyrics_cleaned',right_on = "content")

In [24]:
topic_name_map_gpt_5 = {
    "bt-0":  "Rap de rue, débrouille et violence.",
    "bt-1":  "Questionnement intérieur et conflits émotionnels.",
    "bt-2":  "Fête, nuit, ambiance club et euphorie.",
    "bt-3":  "Relations amoureuses compliquées et tension romantique.",
    "bt-4":  "Ambition, réussite personnelle et confiance en soi.",
    "bt-5":  "Critique sociale et dénonciation des injustices.",
    "bt-6":  "Résilience face aux épreuves et à l’adversité.",
    "bt-7":  "Nostalgie, souvenirs et regard sur le passé.",
    "bt-8":  "Violence, confrontation et posture agressive.",
    "bt-9":  "Vie de luxe, argent et réussite matérielle.",
    "bt-10": "Vie urbaine, quartier et identité locale.",
    "bt-11": "Amitié, loyauté et esprit de bande.",
    "bt-12": "Vulnérabilité, chagrin d’amour et manque de l’autre.",
    "bt-13": "Fierté artistique, performance et créativité.",
    "bt-14": "Fatigue, routine et poids du quotidien.",
    "bt-15": "Discipline, progrès personnel et motivation.",
    "bt-16": "Rébellion, provocation et rejet des normes.",
    "bt-17": "Rêves d’évasion, voyage et quête de liberté.",
    "bt-18": "Spiritualité, croyances et questions existentielles.",
    "bt-19": "Compétition, ego et domination lyricale.",
    "bt-20": "Séduction, désir et attirance physique.",
    "bt-21": "Perte, deuil et douleur émotionnelle.",
    "bt-22": "Conflits, trahison et rupture de confiance.",
    "bt-23": "Humour, dérision et récits volontairement absurdes."
}


merged['topic_name_gpt_5'] = merged['topic_id'].map(topic_name_map_gpt_5)

In [28]:
### VISUALISAtIOn


#Visualizing Topic Evolution Across Years (1999-2028)

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

def visualize_topic_evolution(merged, year_column='year'):
    """
    Create interactive time evolution visualization
    """

    

    # Group by year and topic
    topic_year = (
        merged.groupby(['year', 'topic_name_gpt_5'])
        .size()
        .reset_index(name='count')
    )
    # Calculate percentage per year
    year_totals = merged.groupby('year').size().reset_index(name='total')
    topic_year = topic_year.merge(year_totals, on='year')
    topic_year['percentage'] = (topic_year['count'] / topic_year['total']) * 100

    # Create line plot
    fig = px.line(
        topic_year,
        x='year',
        y='count',
        color='topic_name_gpt_5',
        title='Évolution des Topics du Rap Français au Fil du Temps',
        labels={
            'year': 'Année',
            'percentage': 'Pourcentage (%)',
            'topic': 'topic_name_gpt_5'
        },
        hover_data=['count']
    )

    fig.update_layout(
        height=600,
        hovermode='x unified',
        legend=dict(
            orientation="v",
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.05
        )
    )

    fig.write_html('topic_evolution_over_time.html')
    fig.show()

    return topic_year

visualize_topic_evolution(merged[merged.year>1991])

,year,topic_name_gpt_5,count,total,percentage
0,1992.0,"Amitié, loyauté et esprit de bande.",1,29,3.448276
1,1992.0,"Conflits, trahison et rupture de confiance.",2,29,6.896552
2,1992.0,Critique sociale et dénonciation des injustices.,9,29,31.034483
3,1992.0,"Rap de rue, débrouille et violence.",2,29,6.896552
4,1992.0,"Spiritualité, croyances et questions existenti...",1,29,3.448276
...,...,...,...,...,...
710,2025.0,"Spiritualité, croyances et questions existenti...",2,48,4.166667
711,2025.0,"Séduction, désir et attirance physique.",2,48,4.166667
712,2025.0,"Vie de luxe, argent et réussite matérielle.",3,48,6.250000
713,2025.0,"Vie urbaine, quartier et identité locale.",2,48,4.166667


In [33]:
import plotly.express as px

def visualize_topic_share_per_year(merged, year_column='year', topic_column='topic_name_gpt_5'):
    # 1. Get raw counts per (year, topic)
    topic_year = (
        merged.groupby([year_column, topic_column])
        .size()
        .reset_index(name='count')
    )

    # 2. Pivot so columns = topics, index = year
    pivot = topic_year.pivot_table(
        index=year_column,
        columns=topic_column,
        values='count',
        fill_value=0
    )
    # 3. Normalize so each row sums to 1 (get proportions)
    share = pivot.div(pivot.sum(axis=1), axis=0)

    # 4. Reset index for plotting
    share = share.reset_index()

    # 5. Plot: Stacked area (share per topic at each year, sum to 1)
    fig = px.area(
        share,
        x=year_column,
        y=share.columns[1:],  # All topics
        title="Part relative des topics dans le rap français (part de chaque topic, somme=1 chaque année)",
        labels={year_column: "Année", "value": "Part dans le corpus", "variable": "Topic"},
        groupnorm='fraction'   # Ensures area sum = 1 each year
    )
    fig.update_layout(
        yaxis=dict(title="Part dans le corpus", tickformat=".0%"),
        xaxis_title="Année",
        hovermode='x unified'
    )
    fig.show()
    return share

# Usage
final = visualize_topic_share_per_year(merged[(merged.year > 1991) & (merged.year < 2024)])

In [41]:

def visualize_topic_heatmap(merged, year_column='year'):
    """
    Create heatmap of topic prevalence by year
    """

   

    # Create pivot table
    pivot = merged.pivot_table(
        index='topic_name_gpt_5',
        columns='year',
        values='title',
        aggfunc='count',
        fill_value=0
    )

    # Normalize by year (percentage)
    pivot_pct = pivot.div(pivot.sum(axis=0), axis=1) * 100

    # Create heatmap
    fig = go.Figure(data=go.Heatmap(
        z=pivot_pct.values,
        x=pivot_pct.columns,
        y=pivot_pct.index,
        colorscale='Oranges',
        hoverongaps=False,
        colorbar=dict(title="Pourcentage (%)")
    ))

    fig.update_layout(
        title='Heatmap: Prévalence des Topics par Année',
        xaxis_title='Année',
        yaxis_title='Topic',
        height=800
    )

    fig.write_html('topic_heatmap.html')
    fig.show()

    return pivot_pct




visualize_topic_heatmap(merged[(merged.year > 1991) & (merged.year < 2024)], year_column='year')

year,1992.0,1993.0,1994.0,1995.0,1996.0,1997.0,1998.0,1999.0,2000.0,2001.0,...,2014.0,2015.0,2016.0,2017.0,2018.0,2019.0,2020.0,2021.0,2022.0,2023.0
topic_name_gpt_5,,,,,,,,,,,,,,,,,,,,,
"Ambition, réussite personnelle et confiance en soi.",0.000000,1.5625,0.000000,7.344633,9.302326,3.980100,2.439024,7.046980,4.137931,4.532578,...,5.924018,5.023697,3.608247,3.240392,1.931701,2.003578,1.294033,1.677852,1.221805,0.829646
"Amitié, loyauté et esprit de bande.",3.571429,7.8125,10.810811,5.649718,3.488372,4.975124,5.691057,8.724832,6.896552,8.215297,...,4.893754,4.976303,4.166667,5.011304,3.794412,4.042934,3.235083,3.504847,3.712406,3.871681
"Compétition, ego et domination lyricale.",0.000000,0.0000,5.405405,2.824859,3.488372,1.990050,1.897019,1.342282,3.793103,3.399433,...,7.018674,7.440758,6.701031,6.518463,7.519834,6.940966,6.182602,5.369128,4.746241,6.250000
"Conflits, trahison et rupture de confiance.",7.142857,1.5625,8.108108,2.259887,2.325581,8.457711,7.317073,5.704698,4.482759,7.365439,...,7.984546,6.303318,5.627148,5.501130,4.656778,3.792487,3.486700,3.840418,2.913534,3.429204
Critique sociale et dénonciation des injustices.,32.142857,15.6250,13.513514,8.474576,11.627907,8.955224,6.504065,7.046980,6.896552,9.065156,...,6.761108,5.450237,2.920962,2.938960,2.311142,1.359571,2.084831,1.677852,1.409774,1.216814
"Discipline, progrès personnel et motivation.",0.000000,0.0000,0.000000,2.824859,0.000000,0.497512,0.542005,0.671141,1.034483,1.133144,...,5.086929,3.175355,3.865979,3.617182,2.897551,2.254025,2.839684,2.199851,2.020677,3.373894
"Fatigue, routine et poids du quotidien.",0.000000,1.5625,0.000000,0.564972,1.162791,1.990050,2.168022,2.013423,1.034483,1.983003,...,7.018674,8.246445,9.621993,8.477769,6.864436,6.225403,4.565061,4.250559,3.853383,4.314159
"Fierté artistique, performance et créativité.",0.000000,0.0000,0.000000,1.129944,2.325581,0.995025,0.813008,1.006711,1.379310,1.133144,...,2.189311,4.549763,6.572165,6.103994,7.623318,8.872987,7.476636,7.046980,5.874060,7.688053
"Fête, nuit, ambiance club et euphorie.",0.000000,0.0000,2.702703,0.000000,4.651163,2.487562,0.271003,1.006711,1.034483,0.283286,...,3.283967,4.928910,6.142612,6.292389,7.554329,8.443649,7.476636,7.196122,4.558271,6.913717


In [46]:


def visualize_topic_animation(merged, year_column='year'):
    """
    Animated bubble chart showing topic evolution
    """

    # Get topic data
    
    # Group by year and topic
    topic_year = (
        merged.groupby(['year', 'topic_name_gpt_5'])
        .size()
        .reset_index(name='count')
    )

    # Get UMAP coordinates for each topic centroid
    # (This gives X, Y position for visualization)
    topic_coords = merged.groupby('topic_name_gpt_5')[['x', 'y']].mean().reset_index()

    topic_year = topic_year.merge(topic_coords, on='topic_name_gpt_5')

    # Create animated scatter
    fig = px.scatter(
        topic_year,
        x='x',
        y='y',
        size='count',
        color='topic_name_gpt_5',
        animation_frame='year',
        title='Animation: Évolution des Topics (1991-2024)',
        labels={'count': 'Nombre de Chansons'},
        size_max=50
    )

    fig.update_layout(height=600)
    fig.write_html('topic_animation.html')
    fig.show()

    return topic_year

visualize_topic_animation(merged[(merged.year > 1991) & (merged.year < 2024)], year_column='year')

,year,topic_name_gpt_5,count,x,y
0,1992.0,"Amitié, loyauté et esprit de bande.",1,1.442862,5.882367
1,1992.0,"Conflits, trahison et rupture de confiance.",2,-1.333258,5.855244
2,1992.0,Critique sociale et dénonciation des injustices.,9,-2.250336,5.182127
3,1992.0,"Rap de rue, débrouille et violence.",2,-2.480747,6.313889
4,1992.0,"Spiritualité, croyances et questions existenti...",1,0.917244,4.840314
...,...,...,...,...,...
667,2023.0,"Séduction, désir et attirance physique.",129,1.743343,7.104403
668,2023.0,"Vie de luxe, argent et réussite matérielle.",24,-0.146781,4.072182
669,2023.0,"Vie urbaine, quartier et identité locale.",13,-3.282642,5.513839
670,2023.0,"Violence, confrontation et posture agressive.",117,-2.398178,9.068723
